# Pipeline làm sạch dữ liệu

Notebook này **không còn chứa logic**. Toàn bộ 482 dòng làm sạch đã chuyển
nguyên văn sang `gladiators.data.pipeline`, và cả notebook lẫn CLI lẫn CI đều
gọi **cùng một hàm** — nên không có bản thứ hai để trôi khỏi nhau.

Logic không đổi một dòng. Thứ đổi là chỗ nó *sống*: trước đây nó bị khoá trong
hai cell phải bấm Run All bằng tay, nên không CI nào chạy lại được và không có
gì để rollback về.

## Chạy không cần notebook

```bash
# dựng lại và CHỨNG MINH tái lập được (so schema + giá trị với bản đang chạy)
PYTHONPATH=src python scripts/build_dataset.py --verify-against data/processed

# dựng, đặt tên, và trỏ hệ vào bản mới — không cần dựng lại server
PYTHONPATH=src python scripts/build_dataset.py --publish --activate
```


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/raw").is_dir()
)
sys.path.insert(0, str(REPO_ROOT / "src"))

from gladiators.data.pipeline import run_pipeline
from gladiators.data.coverage import write_manifest

INPUT_DIR = REPO_ROOT / "data/raw"
OUTPUT_DIR = REPO_ROOT / "data/processed"
FAIL_ON_ERROR = False


In [ ]:
report = run_pipeline(INPUT_DIR, OUTPUT_DIR)
write_manifest(OUTPUT_DIR)

summary = {
    "status": report["status"],
    "rows": report["rows"],
    "metric_rows": report["metric_rows"],
    "severity_counts": report["severity_counts"],
    "report": (OUTPUT_DIR / "pipeline_report.json").as_posix(),
}
display(summary)

if FAIL_ON_ERROR and report["status"] == "failed":
    raise RuntimeError(
        "Pipeline found data-quality errors; inspect data_quality_issues.csv"
    )
